# Spatial bbox queries — per geometry type

Fixed `N ≈ 50_000` vertices per store across four geometry types
(points, polylines, graph, mesh) all positioned in `[0, 1000)³`.  Sweep
bbox volume fraction `V ∈ {0.001, 0.01, 0.1, 1.0}` of the domain and
time `open_zv(store)[0].filter(bbox=(lo, hi)).compute()` — only the
chunks the bbox intersects are read.

10 runs per `V`; each run draws a fresh random bbox location.  Each
panel also shows the **full-store read time** as a horizontal dashed
reference so you can see how much the spatial index saves at small
`V`.

Runtime: ~5–10 minutes on a laptop (`V = 1.0` over four geometries
dominates).

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points    import write_points
from zarr_vectors.types.polylines import write_polylines
from zarr_vectors.types.graphs    import write_graph
from zarr_vectors.types.meshes    import write_mesh
from zarr_vectors.lazy import open_zv

N           = 50_000            # target vertex count per geometry
V_FRACTIONS = [0.001, 0.01, 0.1, 1.0]
DOMAIN      = 1000.0
CHUNK       = (200.0, 200.0, 200.0)
BIN         = (50.0,  50.0,  50.0)
SEED        = 0

## 2 · Synthetic generators (one per type)

In [ ]:
rng = np.random.default_rng(SEED)


def _points_input():
    return rng.uniform(0, DOMAIN, (N, 3)).astype(np.float32)


def _polylines_input():
    # ~N total vertices spread across short random walks (~12 verts each).
    counts = rng.integers(8, 16, size=N // 12)
    out = []
    for c in counts:
        steps = rng.normal(0, 5, (c, 3))
        start = rng.uniform(0, DOMAIN, 3)
        out.append((start + steps.cumsum(axis=0)).astype(np.float32))
    return (out,)


def _graph_input():
    positions = rng.uniform(0, DOMAIN, (N, 3)).astype(np.float32)
    src = rng.integers(0, N, size=3 * N // 2)
    dst = rng.integers(0, N, size=3 * N // 2)
    mask = src != dst
    edges = np.stack([src[mask], dst[mask]], axis=1).astype(np.int64)
    return positions, edges


def _mesh_input():
    side = int(np.sqrt(N))
    xs, ys = np.meshgrid(
        np.linspace(0, DOMAIN, side), np.linspace(0, DOMAIN, side),
    )
    zs = rng.uniform(0, 100, (side, side))
    verts = np.stack([xs, ys, zs], axis=-1).reshape(-1, 3).astype(np.float32)
    i = np.arange(side - 1)
    j = np.arange(side - 1)
    ii, jj = np.meshgrid(i, j, indexing='ij')
    a = (ii * side + jj).ravel()
    b = a + 1
    c = a + side
    d = c + 1
    faces = np.concatenate([
        np.stack([a, b, c], axis=1),
        np.stack([b, d, c], axis=1),
    ]).astype(np.int64)
    return verts, faces


GEOMETRIES = [
    ('points',    write_points,    _points_input()),
    ('polylines', write_polylines, _polylines_input()),
    ('graph',     write_graph,     _graph_input()),
    ('mesh',      write_mesh,      _mesh_input()),
]

## 3 · Write one store per geometry

In [ ]:
stores = {}
read_all_s = {}

for name, write_fn, args in GEOMETRIES:
    store = _new_store(f'bbox_{name}')
    write_fn(store, *args, chunk_shape=CHUNK, bin_shape=BIN)
    stores[name] = store

    # Full-store baseline read time (average over a few runs) — used as
    # a dashed reference line on each panel.
    t_full = np.array([
        _time(lambda s=store: open_zv(s)[0].vertices.compute())[0]
        for _ in range(3)
    ])
    read_all_s[name] = float(t_full.mean())
    print(f'{name:9s}  store {_store_bytes(store) / 1e6:6.2f} MB  '
          f'read-all {read_all_s[name] * 1e3:6.1f} ms')

## 4 · Run the sweep

In [ ]:
metrics = ['bbox_s', 'returned_count']
# raw[metric][geom] is shape (len(V_FRACTIONS), N_RUNS).
raw = {
    m: {name: np.zeros((len(V_FRACTIONS), N_RUNS))
        for name, _, _ in GEOMETRIES}
    for m in metrics
}

for i, v in enumerate(V_FRACTIONS):
    edge = DOMAIN * (v ** (1 / 3))
    for run in range(N_RUNS):
        lo = rng.uniform(0, DOMAIN - edge, 3).astype(np.float32)
        hi = (lo + edge).astype(np.float32)

        for name, _, _ in GEOMETRIES:
            zv = open_zv(stores[name])
            t, out = _time(
                lambda: zv[0].filter(bbox=(lo, hi)).compute()
            )
            raw['bbox_s'][name][i, run]         = t
            raw['returned_count'][name][i, run] = out['vertex_count']

# Tidy long-form df: one row per (geom, V).
rows = []
for name, _, _ in GEOMETRIES:
    for i, v in enumerate(V_FRACTIONS):
        row = {'geom': name, 'V': v}
        for m in metrics:
            mean, hw = _mean_ci95(raw[m][name][i])
            row[f'{m}_mean'] = round(mean, 4)
            row[f'{m}_hw']   = round(hw,   4)
        rows.append(row)
df = pd.DataFrame(rows)

# Cleanup all stores.
for path in stores.values():
    shutil.rmtree(Path(path).parent, ignore_errors=True)

## 5 · Results

In [ ]:
df

## 6 · Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
COLORS = {'points': 'C0', 'polylines': 'C1', 'graph': 'C2', 'mesh': 'C3'}

for ax, (name, _, _) in zip(axes.flat, GEOMETRIES):
    sub  = df[df['geom'] == name].sort_values('V')
    mean = sub['bbox_s_mean'].values
    hw   = sub['bbox_s_hw'].values
    ax.fill_between(sub['V'], mean - hw, mean + hw,
                    color=COLORS[name], alpha=0.2)
    ax.loglog(sub['V'], mean, 'o-', color=COLORS[name], label='bbox read')
    ax.axhline(read_all_s[name], color='0.4', linestyle='--', linewidth=1,
               label=f'read all ({read_all_s[name] * 1e3:.0f} ms)')
    ax.set_xlabel('bbox volume fraction')
    ax.set_ylabel('seconds')
    ax.set_title(name)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='best', fontsize=8)

fig.suptitle(
    f'Spatial bbox queries by geometry type '
    f'(N ≈ {N:,} verts, {N_RUNS} runs, 95% CI)',
)
plt.tight_layout()